# Notebook 01 — Extraction

Extracts financial covenant terms from each synthetic agreement, using schema-constrained, citation-grounded extraction: every value returned must carry an exact source citation, or be explicitly marked as not found. Nothing is inferred or generated.

**Input:** `synthetic_agreements/*.txt`
**Output:** `records/01_extraction_output.json`

In [1]:
import json
import os
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv()  # searches this notebook's directory and upward for a .env file

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found. Check that .env exists in the project root "
    "and contains a line like ANTHROPIC_API_KEY=sk-ant-..."
)

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY from the environment automatically

AGREEMENTS_DIR = Path("../synthetic_agreements")
RECORDS_DIR = Path("../records")
RECORDS_DIR.mkdir(exist_ok=True)

print("Setup complete. Client initialized, directories ready.")

Setup complete. Client initialized, directories ready.


## Extraction schema

`threshold_tiers` is a list rather than a single value on purpose. Most covenants have exactly one tier — a single threshold that applies from the agreement's closing date onward. Facility C's amendment is the exception: it changes the threshold partway through the facility's life, which extraction must represent as two tiers, each with its own applicable period, rather than overwriting the old value.

In [2]:
COVENANT_EXTRACTION_TOOL = {
    "name": "record_extracted_covenants",
    "description": "Record every financial covenant found in the source document, with every value grounded in an exact source citation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "covenants": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "covenant_name": {
                            "type": "string",
                            "description": "The defined term for this covenant exactly as it appears in the document, e.g. 'Delinquency Ratio' or 'Consolidated Net Leverage Ratio'."
                        },
                        "definition_text": {
                            "type": ["string", "null"],
                            "description": "A concise paraphrase of how this metric is calculated (numerator and denominator), based only on the document's own definition. Null if no governing definition could be located."
                        },
                        "definition_citation": {
                            "type": ["string", "null"],
                            "description": "The exact, verbatim substring from the source document that defines this term. Must be copied exactly, not paraphrased. Null if not found."
                        },
                        "threshold_operator": {
                            "type": ["string", "null"],
                            "description": "The compliance direction, e.g. 'must not exceed' or 'must be greater than or equal to'."
                        },
                        "threshold_tiers": {
                            "type": "array",
                            "description": "One entry per distinct threshold value and the period range it governs. Most covenants have exactly one tier. A covenant amended to change the threshold over time will have more than one.",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "applies_from": {
                                        "type": ["string", "null"],
                                        "description": "Description or date of the earliest period this tier's threshold governs, as stated in the document."
                                    },
                                    "applies_to": {
                                        "type": ["string", "null"],
                                        "description": "Description or date of the last period this tier's threshold governs, or null if open-ended / still current."
                                    },
                                    "threshold_value": {
                                        "type": "string",
                                        "description": "The exact numeric threshold, e.g. '5.00%' or '3.50:1.00'."
                                    },
                                    "citation": {
                                        "type": "string",
                                        "description": "The exact, verbatim substring from the source document stating this threshold value."
                                    }
                                },
                                "required": ["threshold_value", "citation"]
                            }
                        },
                        "supersedes": {
                            "type": ["string", "null"],
                            "description": "If this document amends a covenant from a prior document, a description of what section/covenant it modifies. Null otherwise."
                        },
                        "document_effective_date": {
                            "type": ["string", "null"],
                            "description": "The date this document (or amendment) itself became effective, as stated in the document."
                        },
                        "extraction_notes": {
                            "type": ["string", "null"],
                            "description": "Any ambiguity, inconsistency, or uncertainty worth flagging. Null if none."
                        }
                    },
                    "required": ["covenant_name", "threshold_tiers"]
                }
            }
        },
        "required": ["covenants"]
    }
}

SYSTEM_PROMPT = """You are extracting financial covenant terms from a loan or credit agreement for a portfolio monitoring system. Follow these rules strictly:

1. Extract only. Never infer, estimate, or generate a value that is not explicitly stated in the source text.
2. Every citation field must be an exact, verbatim substring copied from the source document -- not a paraphrase, not a summary, not a reconstruction. If you cannot find an exact matching span, leave the citation null.
3. If a field's value cannot be verified against explicit text in the document, return null for that field rather than guessing.
4. Extract every financial covenant in the document -- a document may contain more than one.
5. Pay close attention to cross-references: a covenant's operative clause and its governing definition are often in different sections. Resolve the full definition before extracting the threshold.
6. If the document is an amendment, extract the covenant as amended, and populate 'supersedes' and 'document_effective_date' accordingly."""

## Extraction call

`tool_choice` forces the model to respond through the schema above rather than free text — this is the mechanism behind the "extractive, not generative" claim: the model cannot return anything that doesn't fit the defined structure.

In [ ]:
def extract_covenants(document_text: str, document_name: str) -> list[dict]:
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        tools=[COVENANT_EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "record_extracted_covenants"},
        messages=[
            {"role": "user", "content": f"Source document: {document_name}\n\n{document_text}"}
        ],
    )
    for block in response.content:
        if block.type == "tool_use":
            covenants = block.input["covenants"]
            while isinstance(covenants, str):
                covenants = json.loads(covenants)
            if isinstance(covenants, dict):
                covenants = covenants.get("covenants", [covenants])
            return covenants
    return []

## Citation verification

This is a real, programmatic check, not something the model self-reports: every citation the model returns is checked to confirm it is an exact substring of the actual source document. If the model paraphrases instead of quoting — even slightly — this check catches it and flags the field as unverified rather than silently trusting it.

In [13]:
def _as_dict(item):
    while isinstance(item, str):
        item = json.loads(item)
    return item

def verify_citations(covenants: list[dict], source_text: str) -> list[dict]:
    covenants = [_as_dict(c) for c in covenants]
    for covenant in covenants:
        def_citation = covenant.get("definition_citation")
        covenant["definition_citation_verified"] = (def_citation in source_text) if def_citation else None
        for tier in covenant.get("threshold_tiers", []):
            tier["citation_verified"] = tier["citation"] in source_text
    return covenants

## Run extraction over all four documents

Facility C contributes two documents — the original agreement and its amendment — extracted independently here. Resolving which one governs at a given date is deliberately left to Notebook 03, not decided here.

In [10]:
documents = {
    "facility_a_credit_agreement.txt": "Facility A",
    "facility_b_credit_agreement.txt": "Facility B",
    "facility_c_credit_agreement.txt": "Facility C (original)",
    "facility_c_amendment_1.txt": "Facility C (Amendment 1)",
}

all_extractions = {}

for filename, label in documents.items():
    text = (AGREEMENTS_DIR / filename).read_text()
    print(f"Extracting from {label} ({filename})...")
    covenants = extract_covenants(text, filename)
    covenants = verify_citations(covenants, text)
    all_extractions[filename] = covenants
    print(f"  -> {len(covenants)} covenant(s) extracted")

output_path = RECORDS_DIR / "01_extraction_output.json"
output_path.write_text(json.dumps(all_extractions, indent=2))
print(f"\nSaved to {output_path}")

Extracting from Facility A (facility_a_credit_agreement.txt)...


AttributeError: 'str' object has no attribute 'get'

In [11]:
import inspect
print(inspect.getsource(extract_covenants))

def extract_covenants(document_text: str, document_name: str) -> list[dict]:
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        tools=[COVENANT_EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "record_extracted_covenants"},
        messages=[
            {"role": "user", "content": f"Source document: {document_name}\n\n{document_text}"}
        ],
    )
    for block in response.content:
        if block.type == "tool_use":
            covenants = block.input["covenants"]
            if isinstance(covenants, str):
                covenants = json.loads(covenants)
            return covenants
    return []



In [12]:
print(type(covenants))
print(type(covenants[0]))
print(covenants[0])

<class 'dict'>


KeyError: 0

In [14]:
print(list(covenants.keys()))

['covenants']
